In [135]:
import pandas as pd
df = pd.read_csv("C:/Users/aniruddh.singh/OneDrive - Prodapt Solutions Private Limited/Documents/Project_prac/customer_features.csv")

In [136]:
pd.to_numeric(df["TotalCharges"], errors="coerce")

df["TotalCharges"] = df["TotalCharges"].replace(' ', 0)
df["TotalCharges"] = df["TotalCharges"].replace(" ", 0)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"]).astype(float)

df.loc[df["TotalCharges"] == 0, "TotalCharges"] = df["MonthlyCharges"]

In [137]:
Target = df["churn_flag"]
X = df.drop("customerID", axis=1)
X = X.drop("Churn", axis=1)
X = X.drop("gender", axis=1)



In [138]:
print(Target.head())

0    0
1    0
2    1
3    0
4    1
Name: churn_flag, dtype: int64


In [139]:
numerical_cols = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "service_count",
    "high_charge_flag",
    "is_long_term_customer",
    "auto_pay_flag",
    "has_streaming_bundle"
]

categorical_cols = [
    "Contract",
    "InternetService"
]

X = df[numerical_cols + categorical_cols]

X = pd.get_dummies(
    X,
    columns=categorical_cols,
    drop_first=False,
    dtype=int
)

In [140]:
print("Feature Matrix Shape:", df.shape)
print("Target Shape:", Target.shape)

print("\nEncoded Feature Columns:")
print(X.columns.tolist())

Feature Matrix Shape: (7043, 35)
Target Shape: (7043,)

Encoded Feature Columns:
['tenure', 'MonthlyCharges', 'TotalCharges', 'service_count', 'high_charge_flag', 'is_long_term_customer', 'auto_pay_flag', 'has_streaming_bundle', 'Contract_Month-to-month', 'Contract_One year', 'Contract_Two year', 'InternetService_DSL', 'InternetService_Fiber optic', 'InternetService_No']


In [141]:
print("\nClass Counts:")
print(Target.value_counts())

print("\nClass Distribution:")
print(Target.value_counts(normalize=True))

print("\nClass Distribution (%):")
print((Target.value_counts(normalize=True) * 100).round(2))


Class Counts:
churn_flag
0    5174
1    1869
Name: count, dtype: int64

Class Distribution:
churn_flag
0    0.73463
1    0.26537
Name: proportion, dtype: float64

Class Distribution (%):
churn_flag
0    73.46
1    26.54
Name: proportion, dtype: float64


In [142]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score
from sklearn.metrics import recall_score
from sklearn.metrics import classification_report
from sklearn.metrics import precision_score
from sklearn.model_selection import cross_val_score

In [143]:
X_train, X_test, y_train, y_test = train_test_split(
    X, Target,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [144]:
model = DecisionTreeClassifier(random_state=42, max_depth=5)
model.fit(X_train, y_train)

dt_pred = model.predict(X_test)
print(classification_report(y_test, dt_pred))
print(confusion_matrix(y_test, dt_pred))

              precision    recall  f1-score   support

           0       0.86      0.85      0.86      1035
           1       0.60      0.63      0.62       374

    accuracy                           0.79      1409
   macro avg       0.73      0.74      0.74      1409
weighted avg       0.79      0.79      0.79      1409

[[878 157]
 [138 236]]


In [145]:

model1 = LogisticRegression(max_iter=1000)

model1.fit(X_train, y_train)
lr_pred = model.predict(X_test)

print(classification_report(y_test, lr_pred))
print(confusion_matrix(y_test, lr_pred))

              precision    recall  f1-score   support

           0       0.86      0.85      0.86      1035
           1       0.60      0.63      0.62       374

    accuracy                           0.79      1409
   macro avg       0.73      0.74      0.74      1409
weighted avg       0.79      0.79      0.79      1409

[[878 157]
 [138 236]]


c:\Users\aniruddh.singh\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [146]:
lr_accuracy = accuracy_score(y_test, lr_pred)
lr_precision = precision_score(y_test, lr_pred)
lr_recall = recall_score(y_test, lr_pred)
lr_f1 = f1_score(y_test, lr_pred)

dt_accuracy = accuracy_score(y_test, dt_pred)
dt_precision = precision_score(y_test, dt_pred)
dt_recall = recall_score(y_test, dt_pred)
dt_f1 = f1_score(y_test, dt_pred)

lr_cv_f1 = cross_val_score(
    model1,
    X,
    Target,
    cv=5,
    scoring="f1"
).mean()

dt_cv_f1 = cross_val_score(
    model,
    X,
    Target,
    cv=5,
    scoring="f1"
).mean()

comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree"
    ],
    "Accuracy": [
        lr_accuracy,
        dt_accuracy
    ],
    "Precision (Churned)": [
        lr_precision,
        dt_precision
    ],
    "Recall (Churned)": [
        lr_recall,
        dt_recall
    ],
    "F1 (Churned)": [
        lr_f1,
        dt_f1
    ],
    "CV F1": [
        lr_cv_f1,
        dt_cv_f1
    ]
})

print("\nComparison Table")
print(comparison)

best_model = comparison.loc[
    comparison["F1 (Churned)"].idxmax()
]

print("\nBest Model Based on F1 Score:")
print(best_model["Model"])

c:\Users\aniruddh.singh\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\aniruddh.singh\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown


Comparison Table
                 Model  Accuracy  Precision (Churned)  Recall (Churned)  \
0  Logistic Regression  0.790632             0.600509          0.631016   
1        Decision Tree  0.790632             0.600509          0.631016   

   F1 (Churned)     CV F1  
0      0.615385  0.566347  
1      0.615385  0.571041  

Best Model Based on F1 Score:
Logistic Regression


c:\Users\aniruddh.singh\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [149]:
import os
import joblib
os.makedirs("models", exist_ok=True)

joblib.dump(
    model1,
    "models/logistic_churn.pkl"
)

joblib.dump(
    model,
    "models/tree_churn.pkl"
)

print("\nModels saved successfully!")


Models saved successfully!
